# Validation: quantifying error in detection, calibration, and clustering

*Using Projective Transformation for the Spatial Analysis of Team Behaviors in Football*

The other notebooks (`00`-`03`) take the pipeline's output at face value. This one instead
measures how much to trust it, for the thesis's validation/methodology section, on the three
places error actually enters the pipeline:

1. **Detection (YOLO)** - precision/recall/F1/mean IoU against a small hand-labeled sample of frames.
2. **Pitch calibration (homography)** - reprojection error on landmark points *held out* of the fit,
   not the ones it was fit from (which always look good and don't reflect accuracy elsewhere on the pitch).
3. **Team clustering (K-means)** - accuracy against a small hand-labeled sample of tracks' true team.

None of these has a pre-existing ground-truth split for this project's own broadcast footage, so
every section below hand-labels a *small* sample directly in the notebook (via the same
hover-to-read-pixel-coordinates pattern notebook `00` uses for manual calibration) - enough to
report a defensible number, not to replace a proper evaluation dataset.

## 1. Clone the repository and install dependencies

In [ ]:
import os
import sys

REPO_URL = "https://github.com/Batomet/Magisterka.git"
BRANCH = "claude/field-position-detection-dqda1g"
REPO_DIR = "/content/Magisterka"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull
!git log -1 --oneline

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR + "/src")

In [ ]:
!pip install -q -r requirements.txt

## 2. Mount Google Drive and pick a clip

In [ ]:
from pitchvision import DriveConfig, mount_drive

mount_drive()

# Adjust `root` if BuildingAction/Goals/SetPieces don't live directly under My Drive.
drive_cfg = DriveConfig(root="/content/drive/MyDrive/Magisterka")
print(drive_cfg.building_action_path)
print(drive_cfg.goals_path)
print(drive_cfg.set_pieces_path)

In [ ]:
from pitchvision import VideoFrames, list_videos

goal_videos = list_videos(drive_cfg.goals_path)
print(f"Found {len(goal_videos)} videos in Goals/")

sample_video = goal_videos[0]
frames = VideoFrames(sample_video)
print(sample_video, "-", frames.frame_count, "frames @", frames.fps, "fps")

## 3. Detection accuracy (YOLO)

Pick a handful of frames spread across the clip, hand-label their true player/goalkeeper/referee/ball
boxes, and compare against the detector's own predictions on those exact frames with
`compute_detection_metrics` - an IoU-matched precision/recall/F1/mean-IoU, per class and overall.
Uses the specialized 4-class checkpoint (same one `TrackingPipeline` should be run with) rather than
the generic COCO model, since that's what the thesis pipeline actually relies on.

In [ ]:
from pitchvision import (
    PlayerBallDetector,
    SPORTS_DETECTION_CLASSES,
    download_player_detection_weights,
)

player_weights_path = download_player_detection_weights(
    "/content/drive/MyDrive/pitchvision_models/football-player-detection.pt"
)
detector = PlayerBallDetector(
    weights=player_weights_path, confidence=0.3, classes=SPORTS_DETECTION_CLASSES, imgsz=1280
)

# Spread the sample across the clip rather than clustering near frame 0.
EVAL_FRAME_INDICES = sorted({0, frames.frame_count // 3, 2 * frames.frame_count // 3})
eval_frames = {i: frames.read_frame(i) for i in EVAL_FRAME_INDICES}
frame_detections = {i: detector.detect(frame) for i, frame in eval_frames.items()}

for i, dets in frame_detections.items():
    print(f"frame {i}: {len(dets)} detections")

Hover over each frame below to read off pixel coordinates for every player/goalkeeper/referee/ball
box actually visible in it - fill those into `ground_truth_boxes` in the next cell. You don't need
to label every single player; labeling most of what's clearly visible per frame is enough for a
reasonable precision/recall estimate.

In [ ]:
import cv2
import plotly.express as px

for i, frame in eval_frames.items():
    fig = px.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    fig.update_layout(title=f"Frame {i} - hover to read box corner pixel coordinates", height=700)
    fig.show()

In [ ]:
from pitchvision import compute_detection_metrics, ground_truth_boxes_dataframe, predicted_boxes_dataframe

# EDIT THIS: for each frame index above, list every (class_name, x1, y1, x2, y2) box you can
# confidently place by eye. class_name must be one of "player", "goalkeeper", "referee", "ball".
# Do not leave these defaults - they are placeholders and do not correspond to real boxes in your video.
ground_truth_boxes = {
    EVAL_FRAME_INDICES[0]: [
        ("player", 100, 200, 140, 320),
        ("player", 300, 180, 340, 300),
        ("ball", 500, 400, 515, 415),
    ],
}

_PLACEHOLDER = {
    EVAL_FRAME_INDICES[0]: [
        ("player", 100, 200, 140, 320),
        ("player", 300, 180, 340, 300),
        ("ball", 500, 400, 515, 415),
    ],
}
assert ground_truth_boxes != _PLACEHOLDER, (
    "ground_truth_boxes still holds the placeholder values - edit them with real boxes read off "
    "the hover tooltips above before continuing."
)

predicted_df = predicted_boxes_dataframe(frame_detections)
ground_truth_df = ground_truth_boxes_dataframe(ground_truth_boxes)
detection_metrics_df = compute_detection_metrics(predicted_df, ground_truth_df, iou_threshold=0.5)
detection_metrics_df

## 4. Homography calibration accuracy

`PitchCalibrator.reprojection_error` alone measures error on the *same* points a homography was
fit from - always looks good, and says nothing about accuracy elsewhere on the pitch. Instead,
hand-pick more landmark correspondences than the minimum 4 a fit needs (10-15), and
`compute_calibration_holdout_error` repeatedly fits on a random subset and measures error on the
rest - genuinely held-out reprojection error, pooled over many random splits.

In [ ]:
first_frame = frames.read_frame(0)

fig = px.imshow(cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB))
fig.update_layout(title="Hover to read pixel coordinates for the landmarks below", height=700)
fig.show()

from pitchvision import PITCH_LANDMARKS_M
print("Available landmark names:", list(PITCH_LANDMARKS_M.keys()))

In [ ]:
import numpy as np

from pitchvision import PitchCalibrator, compute_calibration_holdout_error

# EDIT THIS: at least 10 landmarks, read off the hover tooltip above for frame 0. More points and a
# wider spread across the pitch give a more reliable holdout-error estimate. Do not leave these
# defaults - they are placeholders and do not correspond to real points in your video.
landmark_pixels = {
    "top_left_corner": (50, 60),
    "top_right_corner": (1200, 55),
    "bottom_left_corner": (10, 650),
    "bottom_right_corner": (1250, 640),
    "centre_spot": (630, 340),
    "centre_top": (630, 55),
    "centre_bottom": (630, 650),
    "left_penalty_top": (150, 200),
    "left_penalty_bottom": (150, 480),
    "left_penalty_spot": (220, 340),
    "left_six_yard_top": (80, 260),
    "left_six_yard_bottom": (80, 420),
}

_PLACEHOLDER = {
    "top_left_corner": (50, 60),
    "top_right_corner": (1200, 55),
    "bottom_left_corner": (10, 650),
    "bottom_right_corner": (1250, 640),
    "centre_spot": (630, 340),
    "centre_top": (630, 55),
    "centre_bottom": (630, 650),
    "left_penalty_top": (150, 200),
    "left_penalty_bottom": (150, 480),
    "left_penalty_spot": (220, 340),
    "left_six_yard_top": (80, 260),
    "left_six_yard_bottom": (80, 420),
}
assert landmark_pixels != _PLACEHOLDER, (
    "landmark_pixels still holds the placeholder values - edit them with real pixel coordinates "
    "read off the hover tooltip above before continuing."
)

pixel_points = np.array(list(landmark_pixels.values()))
pitch_points = np.array([PITCH_LANDMARKS_M[name] for name in landmark_pixels])

holdout_result = compute_calibration_holdout_error(pixel_points, pitch_points, n_repeats=30)
print(f"{holdout_result['n_points']} points, fit on {holdout_result['n_fit']} each of "
      f"{holdout_result['n_repeats']} random splits, held out on the rest:")
print(f"  mean holdout error: {holdout_result['mean_error_m']:.3f} m")
print(f"  std  holdout error: {holdout_result['std_error_m']:.3f} m")
print(f"  max  holdout error: {holdout_result['max_error_m']:.3f} m")

# The calibrator actually used for tracking below is fit on ALL the points (the holdout evaluation
# above only ever fits on a subset, to fairly measure accuracy on points it hasn't seen).
calibrator = PitchCalibrator.from_point_pairs(pixel_points, pitch_points)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.hist(holdout_result["holdout_errors_m"], bins=15)
plt.xlabel("Held-out reprojection error (m)")
plt.ylabel("Count")
plt.title("Distribution of held-out calibration error across random splits")
plt.show()

## 5. Team-colour clustering accuracy (K-means)

Fit `TeamClassifier` on this clip as usual, run a short tracked window, then hand-label a sample of
*tracks'* true team (by eye, from their jersey) and compare against the resolved cluster `team_id`
with `compute_clustering_accuracy` - accuracy under the best possible cluster-id-to-team matching
(cluster ids are arbitrary, so this can't just be compared directly).

In [ ]:
from pitchvision import TeamClassifier, collect_jersey_colors

jersey_colors = collect_jersey_colors(sample_video, detector, class_names=("player",), stride=30)
team_classifier = TeamClassifier(n_clusters=2).fit(jersey_colors)
print("cluster_swatches (RGB):", team_classifier.cluster_swatches)

In [ ]:
from pitchvision import PlayerTracker, TrackingPipeline

tracker = PlayerTracker(
    weights=player_weights_path, confidence=0.3, classes=SPORTS_DETECTION_CLASSES, imgsz=1280
)
pipeline = TrackingPipeline(
    tracker=tracker,
    calibrator=calibrator,
    team_classifier=team_classifier,
    team_eligible_class_names=("player",),
)
eval_tracks_df = pipeline.run(sample_video, max_frames=90)
eval_tracks_df["class_name"].value_counts()

In [ ]:
# One representative crop per track (its first-seen frame + box), for a quick by-eye team check.
N_TRACKS_TO_LABEL = 16

player_rows = eval_tracks_df[eval_tracks_df["class_name"] == "player"]
first_seen = player_rows.sort_values("frame").drop_duplicates("track_id")
sample_tracks = first_seen.head(N_TRACKS_TO_LABEL)

crops, track_ids = [], []
for _, row in sample_tracks.iterrows():
    frame = frames.read_frame(int(row["frame"]))
    x1, y1, x2, y2 = int(row["bbox_x1"]), int(row["bbox_y1"]), int(row["bbox_x2"]), int(row["bbox_y2"])
    crops.append(frame[max(y1, 0):y2, max(x1, 0):x2])
    track_ids.append(int(row["track_id"]))

n_cols = 4
n_rows = (len(crops) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.5 * n_cols, 2.5 * n_rows))
axes = np.atleast_2d(axes)
for idx in range(n_rows * n_cols):
    row, col = divmod(idx, n_cols)
    ax = axes[row, col]
    ax.set_xticks([])
    ax.set_yticks([])
    if idx >= len(crops):
        ax.axis("off")
        continue
    ax.imshow(cv2.cvtColor(crops[idx], cv2.COLOR_BGR2RGB))
    ax.set_title(f"track {track_ids[idx]}", fontsize=9)
plt.tight_layout()
plt.show()

print("track_ids shown, in order:", track_ids)

In [ ]:
from pitchvision import compute_clustering_accuracy

# EDIT THIS: for as many of the track_ids printed above as you reasonably can (at least a handful),
# the TRUE team by eye ("A" or "B" - which physical team, not a cluster id). Do not leave these
# defaults - they are placeholders.
true_team_by_track = {
    track_ids[0]: "A",
    track_ids[1]: "A",
    track_ids[2]: "B",
    track_ids[3]: "B",
}

_PLACEHOLDER = {
    track_ids[0]: "A",
    track_ids[1]: "A",
    track_ids[2]: "B",
    track_ids[3]: "B",
}
assert true_team_by_track != _PLACEHOLDER, (
    "true_team_by_track still holds the placeholder values - label the tracks shown above (or at "
    "least a handful of them) before continuing."
)

labeled_tracks = eval_tracks_df[eval_tracks_df["track_id"].isin(true_team_by_track)].drop_duplicates("track_id")
true_labels = labeled_tracks["track_id"].map(true_team_by_track)
predicted_labels = labeled_tracks["team_id"]

clustering_result = compute_clustering_accuracy(true_labels, predicted_labels)
print(f"accuracy: {clustering_result['accuracy']:.3f} ({clustering_result['n_samples']} labeled tracks)")
print("cluster id -> true team:", clustering_result["cluster_to_true_label"])
clustering_result["confusion_matrix"]

## Summary

Three numbers to report together in the thesis's validation section:

- **Detection**: `detection_metrics_df` - precision/recall/F1/mean IoU, per class and overall.
- **Calibration**: `holdout_result["mean_error_m"]` (± `std_error_m`) - held-out reprojection error in metres.
- **Clustering**: `clustering_result["accuracy"]` - team-assignment accuracy against hand-labeled tracks.

All three came from a small hand-labeled sample on ONE clip - re-run this notebook against a couple
more clips (ideally one from each of `BuildingAction`/`Goals`/`SetPieces`) before treating any of
these numbers as representative of the pipeline as a whole.